# NumPy NN - NumPy Foundations

> **MLCourse · Data Science Foundations · 01_numpy**

NumPy is the bedrock of the entire Python data stack - this notebook builds the core mental model
of the ndarray: creating, inspecting, slicing, reshaping, broadcasting and summarizing arrays,
which is everything pandas, scikit-learn and deep-learning frameworks quietly depend on.

## What you'll learn

- Why NumPy exists (speed, memory, ecosystem) and how it beats plain Python lists
- Creating arrays with `np.array`, `zeros`, `ones`, `full`, `arange`, `linspace`, `eye`
- Reading array metadata: `shape`, `ndim`, `size`, `dtype`, `itemsize`, `nbytes`
- Choosing and converting dtypes (and dodging silent overflow bugs)
- Indexing and slicing 1-D and 2-D arrays, including steps and reversals
- Views vs copies - the number-one source of "my array changed mysteriously" bugs (intro)
- Reshaping: `reshape`, `ravel` vs `flatten`, `.T`, and `np.newaxis`
- **Broadcasting**: the three compatibility rules, four worked shape diagrams, one instructive failure
- Elementwise arithmetic and comparison operators
- Aggregations with `axis=` (`sum`, `mean`, `std`, `min/max`, `argmin/argmax`) + NaN-safe variants
- Running statistics with `cumsum` / `cumprod`
- Boolean masking: filter, count, and combine conditions safely with `&`, `|`, `~`

## 0. Setup

One import gives us the whole toolbox. We also grab `sys` and `time` so we can *measure* (not just
claim) NumPy's memory and speed advantages in a moment.

> 💡 **Pro tip:** the alias `np` appears in virtually every data-science codebase, textbook and
Stack Overflow answer on Earth. Adopt it unconditionally - never write `import numpy`.

In [1]:
import sys                       # used below to measure Python-object memory overhead
import time                      # wall-clock timing for our speed teaser

import numpy as np               # THE numerical library - imported as `np` by universal convention

print("NumPy version:", np.__version__)

NumPy version: 2.4.6


## 1. Why NumPy?

Three reasons NumPy dominates scientific computing in Python:

1. **Speed** - core routines are compiled C loops (often SIMD-vectorized) instead of interpreted
   Python bytecode; a single NumPy call replaces millions of Python-level iterations.
2. **Memory** - five million floats live in ONE contiguous 40 MB block, not as five million
   scattered Python objects that each cost ~28 extra bytes plus pointer overhead.
3. **Foundation** - pandas Series/DataFrames, scikit-learn estimators, Matplotlib images and
   PyTorch/TensorFlow tensors all speak "NumPy array" as their common language.

Feel it first, understand it later - here is a tiny race: summing five million numbers.

In [2]:
# --- Build two representations of the SAME data ---------------------------------
n = 5_000_000                    # five million integers
py_list = list(range(n))         # plain Python list: n separate boxed int objects
np_array = np.arange(n)          # NumPy ndarray: same values in ONE dense buffer

# --- Time a pure-Python loop -----------------------------------------------------
start = time.perf_counter()      # high-resolution timer
total_py = 0
for value in py_list:            # 5,000,000 interpreted iterations...
    total_py += value            # ...each with boxing/unboxing + reference-counting work
py_seconds = time.perf_counter() - start

# --- Time the NumPy equivalent ----------------------------------------------------
start = time.perf_counter()
total_np = np_array.sum()        # ONE compiled call over contiguous memory
np_seconds = time.perf_counter() - start

print(f"Python loop : {py_seconds * 1000:9.1f} ms   (sum = {total_py})")
print(f"NumPy .sum(): {np_seconds * 1000:9.3f} ms   (sum = {total_np})")
print(f"Speed-up    : ~{py_seconds / np_seconds:,.0f}x")

Python loop :    1194.0 ms   (sum = 12499997500000)
NumPy .sum():     8.434 ms   (sum = 12499997500000)
Speed-up    : ~142x


> ⚠️ **Common pitfall:** timings measured this way wobble between runs. In Jupyter you get a
statistically robust measurement for free with the built-in `%timeit` magic:

```text
%timeit total = 0
for v in py_list:
    total += v          # vs simply:  %timeit np_array.sum()
```

`%timeit` repeats the snippet many times and reports mean +/- spread. Use it whenever you
benchmark anything in a notebook.

## 2. `ndarray` vs Python list

A Python **list** is a flexible container of *references* to arbitrary objects scattered around
memory. A NumPy **ndarray** ("N-dimensional array") is a *typed, dense, contiguous* block of raw
numbers plus metadata (shape, dtype). Same numbers, radically different layout:

|                | list                     | ndarray                 |
|----------------|--------------------------|-------------------------|
| contents       | any objects, mixed types | one fixed dtype         |
| memory         | pointer table + objects  | single flat buffer      |
| math           | manual loops             | whole-array operations  |

Homogeneity is the superpower: because every slot holds the same type at the same byte-width,
NumPy can jump to element `i` with pure pointer arithmetic and hand entire ranges to optimized C.

In [3]:
small_list = list(range(1_000_000))                   # a million boxed Python int objects
small_arr = np.arange(1_000_000, dtype=np.int64)      # the same million ints in one buffer

list_pointer_table = sys.getsizeof(small_list)        # bytes of the LIST itself (pointers only!)
arr_bytes = small_arr.nbytes                          # bytes holding ALL array elements

print(f"list container alone : {list_pointer_table / 1e6:.1f} MB   (+ ~28 B payload per int!)")
print(f"ndarray total        : {arr_bytes / 1e6:.1f} MB   (numbers included)")

list container alone : 8.0 MB   (+ ~28 B payload per int!)
ndarray total        : 8.0 MB   (numbers included)


In [4]:
mixed_py = [1, 2.5, "three"]                # lists happily mix types...
try_mixed = np.array([1, 2.5, 3])           # ...NumPy must pick ONE common dtype instead
try_text = np.array([1, "two", 3.0])        # int + str + float -> everything becomes a string!
print("int+float ->", try_mixed.dtype, try_mixed)
print("int+str   ->", try_text.dtype, try_text)

int+float -> float64 [1.  2.5 3. ]
int+str   -> <U32 ['1' 'two' '3.0']


> ⚠️ **Common pitfall:** mixing types in `np.array` does NOT raise an error - NumPy silently
coerces everything to one dtype (here: unicode strings). If your numbers suddenly become `<U21`,
look for an accidental `"3"` hiding in your data.

> 💡 **Pro tip:** check `.dtype` immediately after building any array from external data. It is
the cheapest sanity check in all of data science.

## 3. Creating arrays

Beyond converting lists, NumPy ships factory functions for every common need: constant fills,
integer ranges, evenly spaced grids and identity matrices.

In [5]:
# --- From Python lists ------------------------------------------------------------
a = np.array([1, 2, 3, 4])            # 1-D from a flat list
M = np.array([[1, 2, 3],              # NESTED lists -> 2-D; each inner list becomes a ROW
              [4, 5, 6]])
print("a :", a)
print("M :\n", M)

# --- Constant-filled factories ------------------------------------------------------
z = np.zeros((2, 3))                  # shape TUPLE, filled with 0.0 (default float64)
o = np.ones(4)                        # 1-D of four 1.0s
f = np.full((2, 2), 7)                # any constant; stays int because 7 is int
print("zeros:", z.shape, z.dtype)
print("full :\n", f)

# --- Range builders -------------------------------------------------------------------
r1 = np.arange(0, 10, 2)              # arange(start, stop EXCLUSIVE, step)
r2 = np.linspace(0, 1, 5)             # linspace(start, stop INCLUSIVE, COUNT of points)
print("arange(0,10,2) :", r1)
print("linspace(0,1,5):", r2)

# --- Identity matrices ------------------------------------------------------------------
i3 = np.eye(3)                        # 3x3, ones on the main diagonal
print("eye(3):\n", i3)

a : [1 2 3 4]
M :
 [[1 2 3]
 [4 5 6]]
zeros: (2, 3) float64
full :
 [[7 7]
 [7 7]]
arange(0,10,2) : [0 2 4 6 8]
linspace(0,1,5): [0.   0.25 0.5  0.75 1.  ]
eye(3):
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


In [6]:
# The classic arange-vs-linspace surprise: floating-point steps + EXCLUSIVE stop
bad = np.arange(0, 1, 0.2)            # step 0.2 -> 1.0 is NEVER reached (and float drift accrues)
good = np.linspace(0, 1, 5)           # count-based -> hits both endpoints exactly
print("arange(0, 1, 0.2)  :", bad, "-> length", bad.size)
print("linspace(0, 1, 5)  :", good, "-> length", good.size)

arange(0, 1, 0.2)  : [0.  0.2 0.4 0.6 0.8] -> length 5
linspace(0, 1, 5)  : [0.   0.25 0.5  0.75 1.  ] -> length 5


> 💡 **Pro tip:** rule of thumb - use `arange` with INTEGER steps, use `linspace` when you care
about endpoints or are stepping by floats. `linspace`'s cousin `endpoint=False` drops the last
point when you want arange-like behaviour but exact spacing.

> ⚠️ **Common pitfall:** `np.zeros(3)` gives a 1-D vector; `np.zeros((3,))` is identical;
`np.zeros((3, 4))` is 2-D. The shape is always a tuple - forgetting the inner parentheses is a
rite of passage.

## 4. Inspecting arrays: the metadata attributes

Every ndarray carries a small header describing its contents. These six attributes answer 90% of
"what do I actually have?" questions during debugging:

| attribute   | meaning                                  |
|-------------|------------------------------------------|
| `shape`     | length along each dimension (tuple)      |
| `ndim`      | number of dimensions (axes)              |
| `size`      | total element count = product of shape   |
| `dtype`     | the element type (see next section)      |
| `itemsize`  | bytes per element                        |
| `nbytes`    | total bytes = size x itemsize            |

In [7]:
T = np.arange(24).reshape(2, 3, 4)    # a 3-D tensor: 2 blocks, each 3x4

print("shape    :", T.shape)         # (2, 3, 4) -> lengths along each axis
print("ndim     :", T.ndim)          # 3 axes
print("size     :", T.size)          # 2*3*4 = 24 elements
print("dtype    :", T.dtype)         # int32 or int64 depending on platform (see below)
print("itemsize :", T.itemsize)       # 4 or 8 bytes per element
print("nbytes   :", T.nbytes)        # size * itemsize bytes in RAM

assert T.size == 24 and T.nbytes == T.size * T.itemsize   # the relationships hold by construction

shape    : (2, 3, 4)
ndim     : 3
size     : 24
dtype    : int64
itemsize : 8
nbytes   : 192


## 5. A tour of dtypes

The dtype decides how many bytes each element uses and how arithmetic behaves. The everyday cast:

| dtype      | bytes | typical use                                        |
|------------|-------|----------------------------------------------------|
| `int32`    | 4     | counters, indices where range < 2 billion          |
| `int64`    | 8     | general-purpose integers (most common default)     |
| `float32`  | 4     | images, GPU/deep-learning weights                  |
| `float64`  | 8     | default float; scientific computing workhorse      |
| `bool`     | 1     | masks - the backbone of filtering (last section!)  |

Converting is done with `.astype(target)` - and it ALWAYS returns a new array (a copy).

In [8]:
i32 = np.array([1, 2, 3], dtype=np.int32)         # explicitly narrow integers
i64 = np.array([1, 2, 3], dtype=np.int64)         # explicitly wide integers
f32 = np.array([1.5, 2.5], dtype=np.float32)      # half-width floats
f64 = np.array([1.5, 2.5], dtype=np.float64)      # default double precision
b = np.array([True, False, True])                 # booleans are first-class citizens

print(i32.dtype, i64.dtype, f32.dtype, f64.dtype, b.dtype)

# --- astype conversions ---------------------------------------------------------
ints_from_floats = np.array([1.9, -2.7, 3.0]).astype(np.int32)   # truncates toward ZERO
bools_from_ints = np.array([0, 1, 99]).astype(bool)              # nonzero -> True
floats_from_bools = b.astype(np.float64)                         # True/False -> 1.0/0.0
print("float->int :", ints_from_floats)          # [ 1 -2  3]  (not rounded!)
print("int->bool  :", bools_from_ints)           # [False  True  True]
print("bool->float:", floats_from_bools)

# --- precision loss when narrowing floats ----------------------------------------
pi32 = np.float32(np.pi)                          # float32 keeps only ~7 decimal digits
print("pi as float64:", np.float64(np.pi))
print("pi as float32:", pi32)                     # ...1415927 <- digits silently dropped

int32 int64 float32 float64 bool
float->int : [ 1 -2  3]
int->bool  : [False  True  True]
bool->float: [1. 0. 1.]
pi as float64: 3.141592653589793
pi as float32: 3.1415927


> ⚠️ **Common pitfall (the big one): FIXED-WIDTH INTEGER OVERFLOW.** An `int8` can only store
-128..127. Arithmetic that leaves that range wraps around SILENTLY - no exception, just wrong
answers. This bites everyone who works with `uint8` image pixels at least once.

In [9]:
tiny = np.array([100, 120], dtype=np.int8)        # two innocent-looking small integers
wrapped = tiny + tiny                             # exact sums would be 200 and 240... impossible!
print("int8 100+120 style wrap ->", wrapped)      # [-56 -16]: values wrapped modulo 256

pixels = np.full((2, 2), 250, dtype=np.uint8)     # uint8: 0..255, the classic image dtype
brighter = pixels + np.full((2, 2), 10, dtype=np.uint8)   # 250 + 10 = 260 -> wraps to 4
print("uint8 250+10 ->", brighter[0, 0])          # 4! A "brighter" image just went almost black.

int8 100+120 style wrap -> [-56 -16]
uint8 250+10 -> 4


> 💡 **Pro tip:** before doing arithmetic on integer arrays, upcast deliberately:
`img.astype(np.int32)` first, compute freely, then clip and downcast back at the very end.

## 6. Indexing & slicing (1-D and 2-D)

NumPy slicing mirrors Python lists - `start:stop:step`, stop always EXCLUSIVE - extended with a
comma syntax for multiple dimensions: `M[row_selector, column_selector]`.

In [10]:
x = np.arange(10, 20)                 # [10 11 12 13 14 15 16 17 18 19]

print("x[0]     ->", x[0])            # first element
print("x[-1]    ->", x[-1])           # last element (negative = count from the end)
print("x[2:5]   ->", x[2:5])          # slice, stop EXCLUSIVE -> indices 2,3,4
print("x[:3]    ->", x[:3])           # everything before index 3
print("x[7:]    ->", x[7:])           # everything from index 7 onward
print("x[::2]   ->", x[::2])          # every 2nd element starting at 0
print("x[::-1]  ->", x[::-1])         # REVERSED copy-view of the whole array

x[0]     -> 10
x[-1]    -> 19
x[2:5]   -> [12 13 14]
x[:3]    -> [10 11 12]
x[7:]    -> [17 18 19]
x[::2]   -> [10 12 14 16 18]
x[::-1]  -> [19 18 17 16 15 14 13 12 11 10]


In [11]:
G = np.arange(1, 13).reshape(3, 4)    # a tidy 3-row x 4-column grid to poke at
print("G:\n", G)

print("G[1, 2]     ->", G[1, 2])      # row 1, col 2 (zero-based) -> 7
print("G[1]        ->", G[1])          # whole row 1 (same as G[1, :])
print("G[:, 1]     ->", G[:, 1])       # whole COLUMN 1 -> [ 2  6 10]
print("G[0:2, 1:3] ->")                # submatrix: rows 0-1, cols 1-2
print(G[0:2, 1:3])
print("G[::-1]     :")                 # rows reversed (upside down)
print(G[::-1])
print("G[::-1, ::-1]:")                # BOTH axes reversed = 180-degree rotation
print(G[::-1, ::-1])

G:
 [[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
G[1, 2]     -> 7
G[1]        -> [5 6 7 8]
G[:, 1]     -> [ 2  6 10]
G[0:2, 1:3] ->
[[2 3]
 [6 7]]
G[::-1]     :
[[ 9 10 11 12]
 [ 5  6  7  8]
 [ 1  2  3  4]]
G[::-1, ::-1]:
[[12 11 10  9]
 [ 8  7  6  5]
 [ 4  3  2  1]]


> ⚠️ **Common pitfall:** `G[1][2]` technically works (index row 1, then index into that row) but
> creates an intermediate view per bracket pair. Always prefer the single-bracket form `G[1, 2]`.

> 💡 **Pro tip:** read `G[:, 1]` out loud as "all rows, column 1". Saying the selector in English
prevents most axis mix-ups.

## 7. Views vs copies - a first look

**Slicing does NOT duplicate data.** A slice returns a *view*: a new array object that looks at
the SAME underlying buffer. This makes slicing free (no copying gigabytes) - but modifying a
view modifies the original. This is the number-one "spooky action" bug for newcomers.

In [12]:
a = np.arange(5)                      # [0 1 2 3 4]
v = a[1:4]                            # a VIEW onto elements 1..3 - no memory copied
v[0] = 999                            # write through the view...
print(a)                              # ...and the ORIGINAL changed too!

safe = a[1:4].copy()                  # .copy() is the escape hatch: real, independent data
safe[0] = -1
print(a)                              # unchanged this time
print("same memory?", np.shares_memory(v, a))     # NumPy can even tell you directly

[  0 999   2   3   4]
[  0 999   2   3   4]
same memory? True


> ⚠️ **Common pitfall:** boolean/fancy indexing (advanced notebook) behaves the OPPOSITE way -
those return copies. Memorize the asymmetry now: **basic slices = views, fancy selections =
copies.** Deep dive with proofs comes in `02_numpy_advanced`.

> 💡 **Pro tip:** when you *want* independence, say so explicitly with `.copy()`. When profiling
reveals copying is slow, reach for views deliberately. Never rely on accident.

## 8. Reshaping, transposing and adding axes

Because data lives in one flat buffer, changing *shape* is usually just rewriting the header -
no data movement at all.

In [13]:
r = np.arange(12)                     # 12 flat values
R = r.reshape(3, 4)                   # reinterpret as 3x4 grid (usually a VIEW)
R_inferred = r.reshape(-1, 4)         # -1 = "you figure this dimension out" -> (3, 4)
flat_again = R.ravel()                # back to 1-D: view IF possible (no copy made)
flat_copy = R.flatten()               # ALWAYS returns a fresh copy

print(R)
print("reshape(-1, 4) matches:", np.array_equal(R, R_inferred))

flat_again[0] = -99                   # mutate via the ravel view...
print(r[0])                           # ...original follows (it was a view)
flat_copy[0] = 12345                  # mutate the flatten COPY...
print(R[0, 0])                        # ...original unaffected (-99 still there)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
reshape(-1, 4) matches: True
-99
-99


In [14]:
C = np.arange(1, 7).reshape(2, 3)     # 2x3 matrix
print("C.T (transpose, 3x2):\n", C.T)             # rows become columns; .T is a view

col_vec = C.T.reshape(-1)[:, np.newaxis]          # hmm, simpler demo below:
x = np.arange(4)                                  # shape (4,)
row_like = x[np.newaxis, :]                       # shape (1, 4) - a 1x4 ROW vector
col_like = x[:, np.newaxis]                       # shape (4, 1) - a 4x1 COLUMN vector
print("x          :", x.shape)
print("x[None, :] :", row_like.shape)
print("x[:, None] :", col_like.shape)

C.T (transpose, 3x2):
 [[1 4]
 [2 5]
 [3 6]]
x          : (4,)
x[None, :] : (1, 4)
x[:, None] : (4, 1)


> 💡 **Pro tip:** `np.newaxis` is literally the same object as `None`; `x[:, None]` is identical
to `x[:, np.newaxis]`. Adding an axis like this is THE standard trick to make vectors broadcast
(next section) into rows or columns.

> ⚠️ **Common pitfall:** `reshape` cannot change the element COUNT: reshaping 12 values into a
(5, 3) raises `ValueError`. The product of the new shape must equal `size`.

## 9. Broadcasting

Broadcasting is NumPy's ruleset for applying binary operators (`+`, `*`, ...) between arrays of
*different* shapes without copying data. The three compatibility rules, applied right-to-left
(from the LAST axis backwards):

1. **Pad**: if one array has fewer dimensions, prepend 1s to its shape until ranks match.
2. **Stretch**: dimensions of size 1 are stretched ("broadcast") along that axis for free.
3. **Fail**: if two dimensions differ and NEITHER is 1 → `ValueError`.

Result shape in each dimension = max of the two padded shapes.

In [15]:
# Example 1 - scalar broadcast: () stretches to anything
v = np.arange(3)                      # shape (3,)
scaled = v * 10                       # scalar 10 has shape (); result (3,)
print("scalar:", scaled.shape, scaled)

# Example 2 - row broadcast: (4,) pads to (1, 4), then stretches DOWN each row
grid = np.ones((3, 4))                # shape (3, 4)
row_bias = np.array([1., 2., 3., 4.]) # shape (4,)  -> treated as (1, 4)
biased = grid + row_bias              # every row gets +[1 2 3 4]
#
#   grid  (3, 4)      row_bias (4,) -> (1, 4)        result (3, 4)
#   [1 1 1 1]         [1 2 3 4]  repeated down       [2 3 4 5]
#   [1 1 1 1]         [1 2 3 4]  repeated down  +    [2 3 4 5]
#   [1 1 1 1]         [1 2 3 4]  repeated down       [2 3 4 5]
print("row-broadcast:\n", biased)

# Example 3 - outer-style: (3, 1) + (3,) -> (3, 3); the spec-classic pairing
col = np.arange(3).reshape(3, 1)      # shape (3, 1): one value per ROW
row = np.array([10, 20, 30])          # shape (3,)  -> padded to (1, 3)
outer = col + row                     # dims: max(3,1)=3 and max(1,3)=3 -> (3, 3)
#
#   col (3,1)        row (3,) stretched right      result (3,3)
#   [[0],            [10 20 30 -> ->               [[10 20 30]
#    [1],                                          [11 21 31]
#    [2]]                                           [12 22 32]]
print("outer-sum:\n", outer)

# Example 4 - stacked stretch: (8, 1, 6, 1) + (7, 1, 5) -> (8, 7, 6, 5)
big = np.zeros((8, 1, 6, 1))          # rank 4
small = np.zeros((7, 1, 5))           # rank 3
result_shape = (big + small).shape    # pad small -> (1, 7, 1, 5); then stretch every 1
#
#   big  (8, 1, 6, 1)      small (7, 1, 5)
#   padded small -> (1, 7, 1, 5)
#   dim-wise max -> (8, 7, 6, 5)
print("(8,1,6,1) + (7,1,5) ->", result_shape)

# Bonus utility: ask NumPy for the broadcast shape WITHOUT doing the math
print("broadcast_shapes:", np.broadcast_shapes((8, 1, 6, 1), (7, 1, 5)))

scalar: (3,) [ 0 10 20]
row-broadcast:
 [[2. 3. 4. 5.]
 [2. 3. 4. 5.]
 [2. 3. 4. 5.]]
outer-sum:
 [[10 20 30]
 [11 21 31]
 [12 22 32]]
(8,1,6,1) + (7,1,5) -> (8, 7, 6, 5)
broadcast_shapes: (8, 7, 6, 5)


In [16]:
# And here is what failure looks like - keep this error message in your head:
F = np.ones((3, 2))                   # shape (3, 2)
g = np.ones(3)                        # shape (3,)  -> padded (1, 3)
try:
    F + g                             # last dims: 2 vs 3, neither is 1 -> boom
except ValueError as err:
    print("ValueError:", err)

ValueError: operands could not be broadcast together with shapes (3,2) (3,) 


> ⚠️ **Common pitfall:** `(n,)` is a VECTOR, not a row OR a column until broadcasting pads it.
If you meant "column", be explicit: `x.reshape(-1, 1)` or `x[:, None]`. Silent orientation bugs
produce plausible-looking garbage downstream.

> 💡 **Pro tip:** broadcasting never materializes the stretched copies - the (8,7,6,5) example
above allocated only 1680 result elements, not 1680 copies of the inputs. That's why it's fast.

## 10. Elementwise arithmetic & comparisons

All standard Python operators act ELEMENTWISE on arrays - every element combined with its
positional partner. Note carefully: `*` is elementwise multiply, NOT matrix multiplication
(that is the `@` operator, coming in the advanced notebook).

In [17]:
u = np.array([1., 2., 3., 4.])
w = np.array([10., 20., 30., 40.])

print("add      :", u + w)            # pairwise sums
print("subtract :", w - u)
print("multiply :", u * w)            # ELEMENTWISE product, not dot product!
print("divide   :", w / u)
print("power    :", u ** 2)           # squares each element
print("mod      :", w % 3)            # remainder per element

# Scalars participate through broadcasting (previous section):
print("u * 2 + 1:", u * 2 + 1)

# Comparisons return BOOLEAN arrays - the raw material for masking:
temps = np.array([18, 21, 25, 29, 31, 27, 22])
print("temps >= 25 ->", temps >= 25)
print("temps == 21 ->", temps == 21)

add      : [11. 22. 33. 44.]
subtract : [ 9. 18. 27. 36.]
multiply : [ 10.  40.  90. 160.]
divide   : [10. 10. 10. 10.]
power    : [ 1.  4.  9. 16.]
mod      : [1. 2. 0. 1.]
u * 2 + 1: [3. 5. 7. 9.]
temps >= 25 -> [False False  True  True  True  True False]
temps == 21 -> [False  True False False False False False]


## 11. Aggregations: collapsing arrays to summaries

Reductions like `sum`, `mean`, `std`, `min`, `max` shrink arrays to scalars - unless you pass
`axis=`, which collapses ONE dimension at a time. Read `axis=` as **"collapse THIS axis"**:

- `axis=0` collapses **rows** → one summary PER COLUMN (result loses the row axis).
- `axis=1` collapses **columns** → one summary PER ROW (result loses the column axis).

In [18]:
A = np.arange(1, 13, dtype=float).reshape(3, 4)   # a 3x4 float grid
print("A:\n", A)

print("total          :", A.sum())            # collapse EVERYTHING -> scalar
print("per-column mean:", A.mean(axis=0))    # (4,) - collapse rows: avg down each column
print("per-row sum    :", A.sum(axis=1))      # (3,) - collapse columns: total of each row

# argmin/argmax return INDEX POSITIONS of extremes (handy for "who won?"):
print("argmin (flat)  :", A.argmin(), "-> element", A.flat[A.argmin()])
print("per-row argmax :", A.argmax(axis=1))   # winning COLUMN index within each row
print("std            :", A.std().round(3))   # population std (ddof=0) - note for stats fans

# keepdims=True keeps collapsed axes as size-1 - CRUCIAL for clean broadcasting later:
print("mean axis=0 keepdims shape:", A.mean(axis=0, keepdims=True).shape)   # (1, 4)

A:
 [[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]]
total          : 78.0
per-column mean: [5. 6. 7. 8.]
per-row sum    : [10. 26. 42.]
argmin (flat)  : 0 -> element 1.0
per-row argmax : [3 3 3]
std            : 3.452
mean axis=0 keepdims shape: (1, 4)


> ⚠️ **Common pitfall:** axis confusion is THE most common numpy bug. Drill the mantra: **axis=0
goes down the rows, axis=1 goes across the columns** - and print `.shape` after every reduction
until it's second nature.

### NaN-aware statistics

One missing reading poisons ordinary reductions (`NaN` propagates through everything). Every
reducer has a NaN-skipping twin prefixed with `nan`.

In [19]:
readings = np.array([[1.0, np.nan, 3.0],          # a sensor grid with a dead pixel
                     [4.0, 5.0, 6.0]])

print("plain sum   :", readings.sum())          # nan - infection spreads
print("nan-safe sum:", np.nansum(readings))     # 19.0 - NaN treated as absent
print("nan-mean    :", np.nanmean(readings))    # average over the 5 valid cells
print("nan-max     :", np.nanmax(readings))

# ### Cumulative operations

sales = np.array([3, 5, 2, 8])                    # units sold per day
print("running totals  :", np.cumsum(sales))    # [ 3  8 10 18]
print("running products:", np.cumprod(sales))   # [  3  15  30 240]

M_cum = np.arange(1, 7).reshape(2, 3)
print("cumsum along rows (axis=1):\n", np.cumsum(M_cum, axis=1))   # running total across each row

plain sum   : nan
nan-safe sum: 19.0
nan-mean    : 3.8
nan-max     : 6.0
running totals  : [ 3  8 10 18]
running products: [  3  15  30 240]
cumsum along rows (axis=1):
 [[ 1  3  6]
 [ 4  9 15]]


## 12. Boolean masking basics

Comparison operators give you boolean arrays; those masks then *select*, *count* and *filter*
data without a single explicit loop. This pattern is arguably the most-used NumPy idiom alive.

In [20]:
day_highs = np.array([18, 21, 25, 29, 31, 27, 22, 17, 30, 26])   # ten daily high temperatures

hot_mask = day_highs > 25                 # elementwise test -> boolean array
print("mask       :", hot_mask)
print("hot days   :", hot_mask.sum(), "of", day_highs.size)       # True counts as 1

hot_values = day_highs[hot_mask]          # filter: keep only positions where mask is True
print("hot values :", hot_values)             # NOTE: indexing with a mask returns a COPY

pleasant = day_highs[(day_highs >= 18) & (day_highs <= 26)]      # combine masks with &
chilly = day_highs[~((day_highs >= 18) & (day_highs <= 26))]     # ~ negates the whole mask
print("pleasant   :", pleasant)
print("not pleasant:", chilly)

mask       : [False False False  True  True  True False False  True  True]
hot days   : 5 of 10
hot values : [29 31 27 30 26]
pleasant   : [18 21 25 22 26]
not pleasant: [29 31 27 17 30]


In [21]:
# Why the parentheses around each condition are MANDATORY: `&` binds TIGHTER than comparisons,
# so the un-parenthesized version is parsed as arr > (3 & arr) < 8 - chained-comparison chaos:
arr = np.arange(1, 10)
try:
    arr > 3 & arr < 8                     # missing parentheses -> disaster at runtime
except ValueError as err:
    print("no parens ->", type(err).__name__, "-", str(err)[:52], "...")

try:
    (arr > 3) and (arr < 8)               # even WITH parens, `and` refuses arrays...
except ValueError as err:
    print("'and'     ->", type(err).__name__, "- truth value of an array is ambiguous")

print("correct   ->", arr[(arr > 3) & (arr < 8)])   # ...elementwise `&` is the answer

no parens -> ValueError - The truth value of an array with more than one eleme ...
'and'     -> ValueError - truth value of an array is ambiguous
correct   -> [4 5 6 7]


> ⚠️ **Common pitfall (top 5 all-time):** forgetting parentheses in compound masks. Write
`(a > 3) & (a < 8)` - always - and use `&`, `|`, `~` instead of Python's `and`, `or`, `not`,
which demand a single truth value and explode on arrays.

> 💡 **Pro tip:** masks are just arrays, so you can build them in named steps for readability:
`is_hot = temps > 25; is_dry = humidity < 40; crisis = temps[is_hot & is_dry]`.

## Summary & key takeaways

- NumPy wins on **speed** (compiled C loops) and **memory** (one dense typed buffer) - it is the
  substrate beneath pandas, scikit-learn and every DL framework.
- Arrays are **homogeneous**: mixed inputs get coerced to one dtype silently - always check
  `.dtype` after ingesting data.
- Know your factories: `array/zeros/ones/full` for constants, `arange` for integer ranges,
  `linspace` for exact endpoints, `eye` for identity.
- Fixed-width integers **overflow silently**; upcast before heavy integer arithmetic.
- Basic slices are **views** (mutations leak through); `.copy()` buys independence.
- `reshape` reinterprets the buffer; `ravel` views-if-possible while `flatten` always copies;
  `[:, None]` adds axes to set up broadcasting.
- **Broadcasting rules**: pad shorter shapes on the left, stretch size-1 axes, otherwise fail.
- `axis=0` collapses rows, `axis=1` collapses columns; use `keepdims=True` to preserve shape for
  follow-up broadcasting; reach for `nan*` twins whenever real-world data might have holes.
- Boolean masks filter and count without loops; combine them with parenthesized `&`, `|`, `~`.

Next stop: **02_numpy_advanced** - fancy indexing, sorting/ranking, joins, linear algebra, the
modern random Generator, and performance engineering.